# Discrete Diffusion (LLaDA-style)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/02_discrete_diffusion.ipynb)

Covers:
- `DiscreteParadigm` — masking, (1/t)-weighted loss, reverse diffusion
- `noise_schedule`: `"linear"` · `"cosine"` · `"sqrt"`
- Bidirectional transformer (`causal=False`)
- Multiple attention variants with the discrete paradigm
- Text generation via iterative unmasking

**Runtime**: GPU (T4 or better)

In [ ]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

In [ ]:
import urllib.request, os
if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt')
    print('Downloaded tiny_shakespeare.txt')

## Basic Setup

`DiscreteParadigm` wraps a **bidirectional** (`causal=False`) `Transformer` and owns:
- the forward masking process  
- the `(1/t)`-reweighted token-prediction loss  
- the reverse diffusion inference loop

In [ ]:
import dantinox as dx
from dantinox.paradigms.diffusion.discrete import DiscreteParadigm, DiscreteConfig

model_cfg = dx.ModelConfig(
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    vocab_size=200, causal=False,   # bidirectional — REQUIRED for diffusion
)
diff_cfg  = DiscreteConfig(noise_schedule='cosine', mask_token_id=4)
paradigm  = DiscreteParadigm(model_cfg, diff_cfg)
print('Paradigm:', paradigm)

In [ ]:
run_dir = dx.Trainer(
    paradigm,
    dx.TrainingConfig(lr=3e-4, epochs=2, batch_size=16),
).fit('tiny_shakespeare.txt')
print('Done:', run_dir)

In [ ]:
import jax, jax.numpy as jnp

model  = dx.load(run_dir, paradigm=paradigm)
prompt = jnp.array([[1, 2, 3, 4, 5]])  # short token-ID prefix

tokens = paradigm.generate(
    model, prompt, jax.random.PRNGKey(42),
    gen_len=64, n_steps=50,
)
print('Generated token IDs:', tokens)

## Noise Schedule Comparison

The noise schedule controls how aggressively tokens are masked as `t → 1`.

| Schedule | `p_mask(t)` | Character |
|---|---|---|
| `linear` | `t` | Uniform masking rate |
| `cosine` | `1 − cos(πt/2)` | Slow start, fast finish |
| `sqrt` | `√t` | Fast start, slow finish |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

t = np.linspace(0, 1, 200)
schedules = {'linear': t, 'cosine': 1 - np.cos(np.pi * t / 2), 'sqrt': np.sqrt(t)}

fig, ax = plt.subplots(figsize=(7, 4))
for name, p in schedules.items():
    ax.plot(t, p, label=name, linewidth=2)
ax.set_xlabel('t (corruption level)')
ax.set_ylabel('p_mask(t)')
ax.set_title('LLaDA Noise Schedules')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Train with each noise schedule and compare loss
schedule_runs = {}
for sched in ('linear', 'cosine', 'sqrt'):
    p  = DiscreteParadigm(
        dx.ModelConfig(dim=128, n_heads=2, head_size=64, num_blocks=2,
                       vocab_size=200, causal=False),
        DiscreteConfig(noise_schedule=sched, mask_token_id=4),
    )
    rd = dx.Trainer(p, dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16)).fit(
        'tiny_shakespeare.txt', run_dir=f'/tmp/dx_disc_{sched}')
    schedule_runs[sched] = rd
    print(f'{sched}: done → {rd}')

## Attention Variants with DiscreteParadigm

The diffusion model's `Transformer` supports the same attention options as the AR one.

In [ ]:
from flax import nnx

def disc_param_count(model_cfg):
    p = DiscreteParadigm(model_cfg, DiscreteConfig(mask_token_id=4))
    m = p.build_model(nnx.Rngs(0))
    return sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(m, nnx.Param)))

disc_attn_cfgs = [
    ('MHA',            dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, causal=False, attention='mha')),
    ('GQA kv_heads=2', dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, causal=False, attention='gqa', kv_heads=2)),
    ('MLA',            dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, causal=False, attention='mla')),
]

for name, cfg in disc_attn_cfgs:
    n = disc_param_count(cfg)
    print(f'{name:20s}  {n/1e6:.2f}M params')

In [ ]:
# Train the discrete model with GQA
gqa_disc = DiscreteParadigm(
    dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                   vocab_size=200, causal=False, attention='gqa', kv_heads=2),
    DiscreteConfig(noise_schedule='cosine', mask_token_id=4),
)
gqa_disc_run = dx.Trainer(
    gqa_disc,
    dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16),
).fit('tiny_shakespeare.txt', run_dir='/tmp/dx_disc_gqa')
print('GQA discrete run:', gqa_disc_run)

## One-liner Discrete Training

`dx.fit('discrete', ...)` is the shortest path to a discrete diffusion checkpoint.

In [ ]:
rd = dx.fit(
    'discrete', 'tiny_shakespeare.txt',
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    vocab_size=200, noise_schedule='cosine', mask_token_id=4,
    lr=3e-4, epochs=2, batch_size=16,
)
print('Discrete run dir:', rd)

## FFN Variants with DiscreteParadigm

In [ ]:
ffn_cfgs = [
    ('SwiGLU MLP',    dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                     vocab_size=200, causal=False, ffn='mlp')),
    ('GELU MLP',      dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                     vocab_size=200, causal=False,
                                     ffn='mlp', ffn_activation='gelu')),
    ('MoE 4 experts', dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                     vocab_size=200, causal=False,
                                     ffn='moe', n_experts=4, top_k=2)),
]

for name, cfg in ffn_cfgs:
    n = disc_param_count(cfg)
    print(f'{name:20s}  {n/1e6:.2f}M params')